In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# 1. ЗАГРУЗКА И РЕКОНСТРУКЦИЯ
print("Loading data...")
train = pd.read_parquet('data/train_solo_track.parquet')
test = pd.read_parquet('data/test_solo_track.parquet')
train = train.sort_values(['route_id', 'timestamp']).reset_index(drop=True)

def fast_reconstruct(series):
    T = series.values.astype(np.float64)
    v30 = np.full(len(T), np.nan)
    zero_idx = np.where(T == 0)[0]
    for i in zero_idx:
        v30[i] = 0
        if i > 0: v30[i-1] = 0
    for _ in range(2):
        for i in range(1, len(v30)):
            if np.isnan(v30[i]) and not np.isnan(v30[i-1]): v30[i] = max(0, T[i] - v30[i-1])
        for i in range(len(v30)-1, 0, -1):
            if np.isnan(v30[i-1]) and not np.isnan(v30[i]): v30[i-1] = max(0, T[i] - v30[i])
    v30[np.isnan(v30)] = T[np.isnan(v30)] / 2
    return v30

tqdm.pandas()
train['t30'] = train.groupby('route_id')['target_1h'].progress_transform(fast_reconstruct)

# Фичи: Лаги и Время
print("Engineering features...")
for l in [1, 2, 48]:
    train[f'lag_{l}'] = train.groupby('route_id')['t30'].shift(l)

t = pd.to_datetime(train['timestamp'])
train['h'] = t.dt.hour
train['m'] = t.dt.minute
train = train.dropna()

feat_cols = ['lag_1', 'lag_2', 'lag_48', 'status_1', 'status_2', 'status_3', 'status_4', 'status_5', 'h', 'm']

# 2. ЦИКЛ ОБУЧЕНИЯ 1000 RIDGE-ЭКСПЕРТОВ
results = []
val_wapes = []
val_start_time = pd.Timestamp("2025-10-31 11:00:00")

print("\nTraining 1000 Stable Ridge Experts...")
pbar = tqdm(train['route_id'].unique())

for rid in pbar:
    r_data = train[train['route_id'] == rid].copy()
    
    # Сплит
    train_part = r_data[r_data['timestamp'] < val_start_time]
    val_part = r_data[r_data['timestamp'] == val_start_time]
    
    if len(val_part) == 0: continue
    
    # Обучаем 8 независимых Ridge (Direct)
    models = []
    for step in range(1, 9):
        y = r_data['t30'].shift(-step).loc[train_part.index].dropna()
        X = train_part.loc[y.index, feat_cols]
        
        model = Ridge(alpha=10.0) # Сильная регуляризация
        model.fit(X, y)
        models.append(model)
        
    # Инференс на Субботу
    last_row = r_data.iloc[-1][feat_cols].values.reshape(1, -1)
    preds_30m = [m.predict(last_row)[0] for m in models]
    preds_30m = np.maximum(0, preds_30m) # Физический пол
    
    # Сборка 1h
    last_t30_fact = r_data.iloc[-1]['t30']
    p1h = [preds_30m[0] + last_t30_fact]
    for i in range(1, 8): p1h.append(preds_30m[i] + preds_30m[i-1])
    
    test_ids = test[test['route_id'] == rid].sort_values('timestamp')['id'].values
    for i in range(8):
        results.append({'id': test_ids[i], 'y_pred': p1h[i]})

# 3. СОХРАНЕНИЕ
sub_df = pd.DataFrame(results).sort_values('id')
sub_df.to_csv('submissions/105_Ridge_Legion_RAW.csv', index=False)

print(f"\nDone! Mean Volume: {sub_df['y_pred'].mean():.2f}")
print(f"Check: {((sub_df['y_pred'].mean()/278500)-1)*100:+.2f}% from target.")

Loading data...


100%|██████████| 1000/1000 [00:14<00:00, 68.51it/s]


Engineering features...

Training 1000 Stable Ridge Experts...


  0%|          | 0/1000 [00:00<?, ?it/s]C:\Users\Николай\PycharmProjects\WBShipment\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
C:\Users\Николай\PycharmProjects\WBShipment\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
C:\Users\Николай\PycharmProjects\WBShipment\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
C:\Users\Николай\PycharmProjects\WBShipment\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
C:\Users\Николай\PycharmProjects\WBShipment\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does no


Done! Mean Volume: 261030.05
Check: -6.27% from target.
